# Tutorial: OmniScore with QCRI OmniScore DeBERTa v3 on Colab

Audience:
- Users who want to score text with a hosted OmniScore model on GPU.

Prerequisites:
- A Colab runtime with GPU enabled.
- `omniscore` installed in the notebook environment.

Learning goals:
- Load a hosted OmniScore checkpoint from Hugging Face.
- Run the documented `QCRI/OmniScore-deberta-v3` example.
- Inspect the returned score names, per-example scores, and overall mean.


## Install

Use one of these install paths in Colab before running the rest of the notebook:

```python
# Public package install
%pip install -q omniscore
```

```python
# Pre-release testing from an uploaded wheel
%pip install -q /content/omniscore-0.1.0-py3-none-any.whl
```

If Colab asks for a runtime restart after installation, restart once and then continue from Step 1.


In [ ]:
import torch

print("Torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Switch Colab to GPU: Runtime > Change runtime type > T4 GPU.")


In [ ]:
import json

from omniscore import OmniScorer, get_example, list_known_models

MODEL_ID = "QCRI/OmniScore-deberta-v3"

assert MODEL_ID in list_known_models()
example = get_example(MODEL_ID)
assert example is not None

scorer = OmniScorer(MODEL_ID, device=device, max_length=128)
result = scorer.score(
    predictions=example.prediction,
    references=example.reference,
    sources=example.source,
    tasks=example.task,
)

payload = {
    "model": MODEL_ID,
    "example": {
        "task": example.task,
        "source": example.source,
        "prediction": example.prediction,
        "reference": example.reference,
    },
    "score_names": list(result.score_names),
    "scores": result.to_list(),
    "mean": result.mean(),
}
print(json.dumps(payload, indent=2))


In [ ]:
import numpy as np

assert result.score_names == (
    "informativeness",
    "clarity",
    "plausibility",
    "faithfulness",
)
assert result.scores.shape == (1, 4)
assert np.all(result.scores >= 1.0)
assert np.all(result.scores <= 5.0)
print("Example completed successfully.")


## Expected result

A tested T4 Colab run returned approximately:

- `informativeness`: `1.456`
- `clarity`: `4.954`
- `plausibility`: `4.949`
- `faithfulness`: `4.863`
- `overall`: `4.056`

The exact values should remain very close for the same hosted checkpoint and dependency stack.
